# HopLab — Motor de análisis

**Uso:** `Runtime → Run all` (Ctrl+F9)  
Al terminar, la última celda imprimirá la URL pública del motor.  
Pégala en la UI (Vercel) en el campo **Conectar motor**.

> Asegúrate de que el runtime tiene **GPU** activa:  
> `Runtime → Change runtime type → T4 GPU`

In [ ]:
# ─── CELDA 0 — CONFIGURACIÓN (única que debes editar) ────────────────────────
# Repo del engine (cámbialo cuando lo subas a GitHub)
REPO_URL    = "https://github.com/TU-USUARIO/TU-REPO.git"
REPO_BRANCH = "master"

# Google Drive: True = persistencia entre sesiones (recomendado)
USE_DRIVE   = True
DRIVE_DATA  = "/content/drive/MyDrive/hoplab-data"  # carpeta base en Drive

# Puerto de la API
API_PORT    = 8000
# ─────────────────────────────────────────────────────────────────────────────

In [ ]:
# ─── CELDA 1 — VERIFICAR GPU ─────────────────────────────────────────────────
import subprocess, sys
try:
    result = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                             "--format=csv,noheader"], capture_output=True, text=True)
    if result.returncode == 0:
        print("✅ GPU detectada:", result.stdout.strip())
    else:
        print("⚠️  Sin GPU — el análisis será más lento (CPU).")
        print("   Ir a: Runtime → Change runtime type → T4 GPU")
except FileNotFoundError:
    print("⚠️  nvidia-smi no disponible — sin GPU.")

# Verificar PyTorch (ya instalado en Colab)
try:
    import torch
    cuda_ok = torch.cuda.is_available()
    print(f"PyTorch {torch.__version__} | CUDA disponible: {cuda_ok}")
except ImportError:
    print("PyTorch no encontrado — se instalará en la siguiente celda.")

In [ ]:
# ─── CELDA 2 — MONTAR DRIVE y CREAR ESTRUCTURA DE CARPETAS ───────────────────
import os, pathlib

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_ROOT = pathlib.Path(DRIVE_DATA)
    print(f"📂 Drive montado. DATA_ROOT = {DATA_ROOT}")
else:
    DATA_ROOT = pathlib.Path("/content/hoplab-data")
    print(f"📂 Usando almacenamiento efímero. DATA_ROOT = {DATA_ROOT}")

for sub in ["videos", "output", "venues", "models", "logs"]:
    (DATA_ROOT / sub).mkdir(parents=True, exist_ok=True)

print("✅ Estructura de carpetas lista:")
for sub in ["videos", "output", "venues", "models"]:
    p = DATA_ROOT / sub
    n = sum(1 for _ in p.iterdir()) if p.exists() else 0
    print(f"   {p}  ({n} elementos)")

In [ ]:
# ─── CELDA 3 — CLONAR / ACTUALIZAR ENGINE (efímero en /content) ─────────────
import subprocess, pathlib

ENGINE_DIR = pathlib.Path("/content/hoplab-engine")

if ENGINE_DIR.exists():
    print("Actualizando engine existente...")
    r = subprocess.run(["git", "-C", str(ENGINE_DIR), "pull", "--ff-only"],
                       capture_output=True, text=True)
    print(r.stdout or r.stderr)
else:
    print(f"Clonando {REPO_URL} (rama {REPO_BRANCH})...")
    r = subprocess.run(["git", "clone", "--branch", REPO_BRANCH,
                        "--depth", "1", REPO_URL, str(ENGINE_DIR)],
                       capture_output=True, text=True)
    print(r.stdout or r.stderr)
    if r.returncode != 0:
        raise RuntimeError(f"Error clonando repo: {r.stderr}")

print(f"✅ Engine listo en {ENGINE_DIR}")

In [ ]:
# ─── CELDA 4 — INSTALAR DEPENDENCIAS ─────────────────────────────────────────
# PyTorch ya viene en Colab con CUDA. Solo instalamos el resto.
import subprocess, sys

REQS = ENGINE_DIR / "hoplab-cloud" / "colab" / "requirements-colab.txt"
if not REQS.exists():
    # Fallback: instalar directamente
    pkgs = ["ultralytics>=8.3", "opencv-python-headless>=4.9",
            "numpy>=1.26", "scipy>=1.13", "matplotlib>=3.9",
            "fastapi>=0.115", "uvicorn>=0.30"]
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + pkgs,
                       capture_output=True, text=True)
else:
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                        "-r", str(REQS)], capture_output=True, text=True)

if r.returncode != 0:
    print("[ERROR pip]:", r.stderr[-2000:])
else:
    print("✅ Dependencias instaladas")

# Cachear pesos YOLO en Drive/models para no descargar cada sesión
import os
os.environ["YOLO_CONFIG_DIR"] = str(DATA_ROOT / "models")
print(f"   Pesos YOLO se cachearán en: {DATA_ROOT / 'models'}")

In [ ]:
# ─── CELDA 5 — CONFIGURAR ENV + SYMLINKS Drive ←→ Engine ─────────────────────
import os, pathlib, sys

# Vars de entorno que api_server.py lee
os.environ["HOPLAB_OUTPUT_ROOT"] = str(DATA_ROOT / "output")
os.environ["HOPLAB_VENUE_ROOT"]  = str(DATA_ROOT / "venues")
os.environ["HOPLAB_VIDEO_ROOT"]  = str(DATA_ROOT / "videos")
os.environ["YOLO_CONFIG_DIR"]    = str(DATA_ROOT / "models")

# Optimización: no escribir frames anotados a Drive (ahorra cuota)
os.environ["TJ_WRITE_ANNOTATED"] = "0"

# Cambiar al directorio del engine para que los imports relativos funcionen
os.chdir(str(ENGINE_DIR))
if str(ENGINE_DIR) not in sys.path:
    sys.path.insert(0, str(ENGINE_DIR))

# Symlinks para compatibilidad con paths que usa el engine directamente
def safe_symlink(src: pathlib.Path, dst: pathlib.Path):
    dst.unlink(missing_ok=True)
    dst.symlink_to(src)
    print(f"   🔗 {dst} → {src}")

print("Creando symlinks...")
safe_symlink(DATA_ROOT / "output", ENGINE_DIR / "output")
safe_symlink(DATA_ROOT / "venues", ENGINE_DIR / "venues")

print("✅ Entorno configurado:")
for k in ["HOPLAB_OUTPUT_ROOT", "HOPLAB_VENUE_ROOT", "HOPLAB_VIDEO_ROOT",
          "TJ_WRITE_ANNOTATED", "YOLO_CONFIG_DIR"]:
    print(f"   {k} = {os.environ[k]}")

In [ ]:
# ─── CELDA 6 — ARRANCAR API ───────────────────────────────────────────────────
import threading, time, urllib.request, urllib.error

def _run_server():
    import uvicorn
    import api_server  # cwd = ENGINE_DIR
    uvicorn.run(api_server.app, host="0.0.0.0", port=API_PORT,
                log_level="warning")

srv_thread = threading.Thread(target=_run_server, daemon=True)
srv_thread.start()

print(f"Esperando API en :{API_PORT} ", end="")
for _ in range(60):
    try:
        urllib.request.urlopen(f"http://127.0.0.1:{API_PORT}/status", timeout=2)
        print(" ✅ API lista")
        break
    except Exception:
        print(".", end="", flush=True)
        time.sleep(2)
else:
    raise RuntimeError("API no respondió en 120s — revisa errores arriba")

In [ ]:
# ─── CELDA 7 — TUNNEL CLOUDFLARED (gratis, sin cuenta) ───────────────────────
import subprocess, threading, re, time, pathlib

CF_DEB = pathlib.Path("/tmp/cloudflared.deb")

if not pathlib.Path("/usr/bin/cloudflared").exists():
    print("Instalando cloudflared...")
    subprocess.run(
        ["wget", "-q", "-O", str(CF_DEB),
         "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb"],
        check=True)
    subprocess.run(["dpkg", "-i", str(CF_DEB)], check=True, capture_output=True)
    print("Cloudflared instalado.")

# Asegurarse de que no hay config.yaml que bloquee los quick tunnels
cf_cfg = pathlib.Path.home() / ".cloudflared" / "config.yaml"
if cf_cfg.exists():
    cf_cfg.rename(cf_cfg.with_suffix(".yaml.bak"))

TUNNEL_URL = None
tunnel_ready = threading.Event()

def _run_tunnel():
    global TUNNEL_URL
    proc = subprocess.Popen(
        ["cloudflared", "tunnel", "--url", f"http://127.0.0.1:{API_PORT}"],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True)
    pattern = re.compile(r"https://[\w-]+\.trycloudflare\.com")
    for line in proc.stdout:
        m = pattern.search(line)
        if m:
            TUNNEL_URL = m.group(0)
            tunnel_ready.set()

t = threading.Thread(target=_run_tunnel, daemon=True)
t.start()

print("Esperando URL del tunnel (máx 60s) ", end="")
ok = tunnel_ready.wait(timeout=60)
if not ok:
    print("\n⚠️  cloudflared no imprimió URL — intenta la celda de fallback debajo")
else:
    print(f"\n✅ Tunnel listo")

In [ ]:
# ─── CELDA 8 — URL PÚBLICA + INSTRUCCIONES ───────────────────────────────────
from IPython.display import display, HTML

if TUNNEL_URL:
    display(HTML(f"""
    <div style="font-family:monospace;background:#1e1e2e;color:#cdd6f4;
                padding:20px;border-radius:12px;font-size:15px;margin:10px 0">
      <b style="font-size:18px;color:#a6e3a1">✅ Motor HopLab listo</b><br><br>

      <b>URL del motor (copiar):</b><br>
      <span style="background:#313244;padding:6px 12px;border-radius:6px;
                   color:#89b4fa;font-size:16px;display:inline-block;margin:8px 0">
        {TUNNEL_URL}
      </span><br><br>

      <b>Siguiente paso:</b><br>
      1. Abre <a href='https://hoplab.vercel.app' style='color:#89dceb'>
            hoplab.vercel.app</a> (o tu URL de Vercel)<br>
      2. Haz clic en <b>Conectar motor</b><br>
      3. Pega la URL de arriba y guarda<br><br>

      <small style='color:#6c7086'>La URL cambia cada sesión de Colab.<br>
      Los análisis se guardan en Drive aunque cierres esta pestaña.</small>
    </div>
    """))
    print(f"\nURL: {TUNNEL_URL}")
    print(f"Docs: {TUNNEL_URL}/docs")
else:
    print("⚠️  Tunnel no disponible. Usa la celda de fallback (localtunnel) abajo.")

In [ ]:
# ─── CELDA 9 — FALLBACK: localtunnel (si cloudflared falla) ─────────────────
# Solo ejecutar si la celda 7 no imprimió URL.
import subprocess, sys

subprocess.run(["npm", "install", "-g", "localtunnel", "--silent"],
               capture_output=True)

lt_proc = subprocess.Popen(
    ["lt", "--port", str(API_PORT)],
    stdout=subprocess.PIPE, text=True)

for line in lt_proc.stdout:
    if "loca.lt" in line:
        lt_url = line.strip().split()[-1]
        print(f"\nFallback URL (localtunnel): {lt_url}")
        print("Nota: localtunnel puede pedir confirmación al abrir en el navegador.")
        break